## Epic 2 — Preprocessing Pipeline

Transforms the raw `MetObjects.csv` into a clean, model-ready feature matrix.
Covers null handling, column selection, and feature engineering for both the supervised (Department classification) and unsupervised (clustering) tracks.

In [1]:
import os

import numpy as np
import pandas as pd
import scipy.sparse
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

In [2]:
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data", "MetObjects.csv")

# Setting low_memory=False forces pandas to read the entire column before deciding its dtype, which is slower but correct.
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print("Loaded:", df_raw.shape)

Loaded: (484956, 54)


In [3]:
# ── Column definitions (single source of truth) ────────────────────────────
# Source: reports/feature_engineering.md

COLS_TO_DROP = [
    # Identifiers
    "Object ID", "Object Number", "Constituent ID",
    # URLs — no feature signal
    "Link Resource", "Artist ULAN URL", "Artist Wikidata URL",
    "Object Wikidata URL", "Tags AAT URL", "Tags Wikidata URL",
    # 100% null / constant
    "Metadata Date", "Repository",
    # Geographic columns >90% null
    "River", "State", "Locus", "County", "Locale",
    "Excavation", "Subregion", "Region", "City",
    # Admin fields with no object-level signal
    "Rights and Reproduction", "Portfolio", "Gallery Number", "Credit Line",
    # Redundant / low-signal artist fields
    "Artist Prefix", "Artist Suffix", "Artist Alpha Sort",
    "Artist Display Bio", "Artist Display Name",
    # Flagged — dropping for now
    "Dynasty", "Reign", "Geography Type", "Country",
    "Artist Begin Date", "Artist End Date",
    "Dimensions", "Title", "Object Date",
]

COLS_NUMERIC = [
    "AccessionYear",          # 0.8% null → fill with median year
]

COLS_CATEGORICAL = [
    "Medium",                 # 1.5% null  — TF-IDF after comma-splitting
    "Tags",                   # 60.3% null — TF-IDF after pipe-splitting
    "Classification",         # 16.2% null — TF-IDF or top-N encode
    "Object Name",            # 0.5% null  — top-N frequency encode
    "Culture",                # 57.1% null — top-100 + "Other"
    "Period",                 # 81.2% null — top-N + "Other"
    "Artist Nationality",     # 41.7% null — top-N + "Other"
    "Artist Role",            # 41.7% null — label encode
    "Artist Gender",          # 78.0% null — binary flag
]

COLS_KEEP = [
    "Department",             # classification target — kept, encoded separately
    "Object Begin Date",      # int64, 0% null — clip + derive features
    "Object End Date",        # int64, 0% null — clip + derive features
    "Is Highlight",           # bool, 0% null
    "Is Timeline Work",       # bool, 0% null
    "Is Public Domain",       # bool, 0% null
]

print("Columns accounted for:",
      len(COLS_TO_DROP) + len(COLS_NUMERIC) + len(COLS_CATEGORICAL) + len(COLS_KEEP),
      "/ 54 total")

Columns accounted for: 54 / 54 total


In [4]:
df = df_raw.drop(columns=COLS_TO_DROP)

print(f"Shape before drop : {df_raw.shape}")
print(f"Shape after drop  : {df.shape}")
print(f"Columns removed   : {df_raw.shape[1] - df.shape[1]}")
print()
assert "Department" in df.columns, "ERROR: Department was accidentally dropped!"
print("Test - Department column intact!!")

Shape before drop : (484956, 54)
Shape after drop  : (484956, 16)
Columns removed   : 38

Test - Department column intact!!


### Missing Value Imputation

In [5]:
# Null counts before imputation — only show columns that have any nulls
null_before = df.isna().sum()
null_before = null_before[null_before > 0].sort_values(ascending=False)

print(f"Columns with nulls: {len(null_before)}")
print(f"Total null cells  : {null_before.sum():,}")
print()
display(null_before.rename("null_count").to_frame())

Columns with nulls: 10
Total null cells  : 1,838,500



,null_count
Period,393813
Artist Gender,378474
Tags,292501
Culture,276766
Artist Role,202443
Artist Nationality,202443
Classification,78717
Medium,7215
AccessionYear,3862
Object Name,2266


In [6]:
COLS_BOOL = ["Is Highlight", "Is Public Domain", "Is Timeline Work"]

# Numeric: coerce to float first (AccessionYear is stored as string in the CSV),
# then fill nulls introduced by coercion with the column median
for col in COLS_NUMERIC:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Categorical: fill with "Unknown" so the model sees a real category, not NaN
for col in COLS_CATEGORICAL:
    df[col] = df[col].fillna("Unknown")

# Booleans: fill with False (absence of a flag is the safe neutral value)
for col in COLS_BOOL:
    df[col] = df[col].fillna(False)

# --- newly added columns ---
df["Period"] = df["Period"].fillna("")              # goes to TF-IDF — empty string, not "Unknown"
df["Artist Nationality"] = df["Artist Nationality"].fillna("Unknown")
df["Artist Role"] = df["Artist Role"].fillna("Unknown")
df["Artist Gender"] = df["Artist Gender"].fillna("Unknown")

### Notes

- AccessionYear is the only numeric column here,
  and year values are likely slightly skewed (e.g.
   a burst of acquisitions in a particular era).
  Mean is pulled by outliers — if a handful of
  records have a weird year like 1800 or 2050, the
   mean shifts. Median is the middle value of the
  sorted column, so outliers can't distort it.
- Top-N frequency encode / label encode:
  "Unknown" becomes its own category. The model
  learns a weight for it — effectively learning "objects with no Culture info tend to land in
  these departments." That's genuinely useful
  signal, because missing = anonymous object is
  not random noise.

In [7]:
remaining_nulls = df.isna().sum().sum()
print(f"Total null cells after imputation: {remaining_nulls}")
assert remaining_nulls == 0, f"Expected 0 nulls, found {remaining_nulls}"
print("Test -- All nulls resolved!")

Total null cells after imputation: 0
Test -- All nulls resolved!


### Simple Column Encoding

In [8]:
# Boolean → int  (True=1, False=0)
COLS_BOOL = ["Is Highlight", "Is Public Domain", "Is Timeline Work"]
for col in COLS_BOOL:
    df[col] = df[col].astype(int)

# AccessionYear was coerced to float in the imputation step; cast to int now
# that nulls are gone (a year like 1978.0 should be stored as 1978)
df["AccessionYear"] = df["AccessionYear"].astype(int)

In [9]:
# Clip date outliers before deriving features
# Values below -7000 are placeholder junk (2,074 records per EDA)
df["Object Begin Date"] = df["Object Begin Date"].clip(lower=-7000, upper=2026)
df["Object End Date"]   = df["Object End Date"].clip(lower=-7000, upper=2026)

# Derive temporal features
# Reference year matches the upper clip so object_age is always >= 0
df["object_age"]  = 2026 - df["Object End Date"]
df["object_span"] = (df["Object End Date"] - df["Object Begin Date"]).clip(lower=0)

In [10]:
COLS_SIMPLE_NUMERIC = COLS_BOOL + ["AccessionYear", "object_age", "object_span"]

assert df[COLS_BOOL].isin([0, 1]).all().all(), "Boolean columns contain values other than 0/1"
assert df["object_age"].min() >= 0,  "object_age has negative values — check clipping"
assert df["object_span"].min() >= 0, "object_span has negative values — check clipping"

print(df[COLS_SIMPLE_NUMERIC].dtypes)

Is Highlight        int64
Is Public Domain    int64
Is Timeline Work    int64
AccessionYear       int64
object_age          int64
object_span         int64
dtype: object


### High-Cardinality Categorical Encoding

In [11]:
# ── Culture: frequency encoding ─────────────────────────────────────────────
# Replace each culture string with how many times it appears in the dataset.
# High-frequency cultures (e.g. "American") get large numbers;
# rare cultures get small ones — the numeric value carries popularity signal.

print(f"Culture unique values BEFORE: {df['Culture'].nunique()}")

culture_freq_map = df["Culture"].value_counts().to_dict()
df["Culture"] = df["Culture"].map(culture_freq_map).fillna(0).astype(int)

print(f"Culture unique values AFTER : {df['Culture'].nunique()}  (now frequency counts)")

Culture unique values BEFORE: 7313
Culture unique values AFTER : 218  (now frequency counts)


In [12]:
# ── Object Name: top-50 encoding ────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

print(f"Object Name unique values BEFORE: {df['Object Name'].nunique()}")

top_object_names = set(df["Object Name"].value_counts().head(50).index)
df["Object Name"] = df["Object Name"].apply(
    lambda x: x if x in top_object_names else "Other"
)

object_name_encoder = LabelEncoder()
df["Object Name"] = object_name_encoder.fit_transform(df["Object Name"])

print(f"Object Name unique values AFTER : {df['Object Name'].nunique()}  (label-encoded integers)")

Object Name unique values BEFORE: 28632
Object Name unique values AFTER : 51  (label-encoded integers)


In [13]:
# ── Classification: top-50 encoding ─────────────────────────────────────────
print(f"Classification unique values BEFORE: {df['Classification'].nunique()}")

top_classifications = set(df["Classification"].value_counts().head(50).index)
df["Classification"] = df["Classification"].apply(
    lambda x: x if x in top_classifications else "Other"
)

classification_encoder = LabelEncoder()
df["Classification"] = classification_encoder.fit_transform(df["Classification"])

print(f"Classification unique values AFTER : {df['Classification'].nunique()}  (label-encoded integers)")

Classification unique values BEFORE: 1245
Classification unique values AFTER : 51  (label-encoded integers)


In [14]:
# ── Artist Nationality: frequency encoding ───────────────────────────────────
print(f"Artist Nationality unique values BEFORE: {df['Artist Nationality'].nunique()}")
nationality_freq_map = df["Artist Nationality"].value_counts().to_dict()
df["Artist Nationality"] = df["Artist Nationality"].map(nationality_freq_map).fillna(0).astype(int)
print(f"Artist Nationality unique values AFTER : {df['Artist Nationality'].nunique()}  (frequency counts)")

print()

# ── Artist Role: top-15 encoding ─────────────────────────────────────────────
print(f"Artist Role unique values BEFORE: {df['Artist Role'].nunique()}")
top_artist_roles = set(df["Artist Role"].value_counts().head(15).index)
df["Artist Role"] = df["Artist Role"].apply(lambda x: x if x in top_artist_roles else "Other")
artist_role_encoder = LabelEncoder()
df["Artist Role"] = artist_role_encoder.fit_transform(df["Artist Role"])
print(f"Artist Role unique values AFTER : {df['Artist Role'].nunique()}  (label-encoded integers)")

print()

# ── Artist Gender: top-3 encoding ────────────────────────────────────────────
print(f"Artist Gender unique values BEFORE: {df['Artist Gender'].nunique()}")
top_artist_genders = set(df["Artist Gender"].value_counts().head(3).index)
df["Artist Gender"] = df["Artist Gender"].apply(lambda x: x if x in top_artist_genders else "Other")
artist_gender_encoder = LabelEncoder()
df["Artist Gender"] = artist_gender_encoder.fit_transform(df["Artist Gender"])
print(f"Artist Gender unique values AFTER : {df['Artist Gender'].nunique()}  (label-encoded integers)")

Artist Nationality unique values BEFORE: 6946
Artist Nationality unique values AFTER : 216  (frequency counts)

Artist Role unique values BEFORE: 7119
Artist Role unique values AFTER : 16  (label-encoded integers)

Artist Gender unique values BEFORE: 291
Artist Gender unique values AFTER : 4  (label-encoded integers)


### TF-IDF Encoding — Medium and Tags

In [15]:
# ── Medium: preprocessing ────────────────────────────────────────────────────
# "Bronze, patinated, gilt" → "Bronze patinated gilt" so the vectorizer
# sees three tokens, not "bronze," with a trailing comma baked in.

df["Medium"] = (
    df["Medium"]
    .fillna("")
    .str.replace(",", " ", regex=False)          # commas → spaces
    .str.replace(r"\s+", " ", regex=True)        # collapse double spaces
    .str.strip()
)

In [16]:
# ── Tags: preprocessing ──────────────────────────────────────────────────────
# "Portraits|Men|Costumes" → "Portraits Men Costumes" so each tag is one token.
# Nulls become "" so the vectorizer produces an all-zero row (not an error).

df["Tags"] = (
    df["Tags"]
    .fillna("")
    .str.replace("|", " ", regex=False)          # pipes → spaces
    .str.strip()
)

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Medium TF-IDF ─────────────────────────────────────────────────────────────
medium_vectorizer = TfidfVectorizer(
    max_features=100,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)
medium_tfidf = medium_vectorizer.fit_transform(df["Medium"])

# ── Tags TF-IDF ───────────────────────────────────────────────────────────────
tags_vectorizer = TfidfVectorizer(
    max_features=50,
    ngram_range=(1, 1),
    sublinear_tf=True,
)
tags_tfidf = tags_vectorizer.fit_transform(df["Tags"])

print(f"medium_tfidf shape : {medium_tfidf.shape}")
print(f"tags_tfidf shape   : {tags_tfidf.shape}")

medium_tfidf shape : (484956, 100)
tags_tfidf shape   : (484956, 50)


In [18]:
print("Top 20 Medium features:")
print(medium_vectorizer.get_feature_names_out()[:20])

print()
print("Top 20 Tags features:")
print(tags_vectorizer.get_feature_names_out()[:20])

Top 20 Medium features:
['albumen' 'albumen photograph' 'albumen silver' 'alloy' 'aquatint'
 'black' 'black chalk' 'black ink' 'blue' 'brass' 'bronze' 'brown'
 'brown ink' 'brush' 'canvas' 'chalk' 'color' 'color lithograph'
 'color paper' 'colored']

Top 20 Tags features:
['abstraction' 'actors' 'actresses' 'and' 'angels' 'animals'
 'architecture' 'arms' 'athletes' 'baseball' 'birds' 'boats' 'boys'
 'buildings' 'carriages' 'children' 'christ' 'coat' 'dogs' 'dragons']


In [19]:
# ── Period: preprocessing ────────────────────────────────────────────────────
# Already null-filled with "" in imputation. Clean up commas and whitespace.
df["Period"] = (
    df["Period"]
    .str.replace(",", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# ── Period TF-IDF ─────────────────────────────────────────────────────────────
period_vectorizer = TfidfVectorizer(
    max_features=50,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words="english",
)
period_tfidf = period_vectorizer.fit_transform(df["Period"])

print(f"period_tfidf shape: {period_tfidf.shape}")
print()
print("Top 15 Period features:")
print(period_vectorizer.get_feature_names_out()[:15])

period_tfidf shape: (484956, 50)

Top 15 Period features:
['1368' '1368 1644' '1615' '1615 1868' '1644' '1644 1911' '1868'
 '1868 1912' '1911' '1912' 'age' 'archaic' 'archaic classical' 'bronze'
 'classical']


### Assemble Final Feature Matrix

In [20]:
# ── Define dense feature columns ─────────────────────────────────────────────
# Exclude: target, text columns already encoded as sparse TF-IDF matrices.
# Artist Nationality, Artist Role, Artist Gender are now numeric (encoded above)
# and will be picked up automatically by the dtype check below.
# Period goes into the sparse hstack, not here.

EXCLUDE = {"Department", "Medium", "Tags", "Period"}

feature_cols = [
    col for col in df.columns
    if col not in EXCLUDE and pd.api.types.is_numeric_dtype(df[col])
]

# Confirm no string columns remain outside EXCLUDE
string_cols_remaining = [
    col for col in df.columns
    if col not in EXCLUDE and not pd.api.types.is_numeric_dtype(df[col])
]

print("Dense feature columns:", feature_cols)
print()
if string_cols_remaining:
    print("WARNING — string columns still not encoded:", string_cols_remaining)
else:
    print("No unencoded string columns remaining ✓")

Dense feature columns: ['Is Highlight', 'Is Timeline Work', 'Is Public Domain', 'AccessionYear', 'Object Name', 'Culture', 'Artist Role', 'Artist Nationality', 'Artist Gender', 'Object Begin Date', 'Object End Date', 'Classification', 'object_age', 'object_span']

No unencoded string columns remaining ✓


In [21]:
from scipy.sparse import csr_matrix, hstack

# Dense columns → sparse (keeps everything in one format for hstack)
X_dense = csr_matrix(df[feature_cols].values)

# Stack horizontally: dense features | medium TF-IDF | tags TF-IDF | period TF-IDF
X = hstack([X_dense, medium_tfidf, tags_tfidf, period_tfidf])

print(f"X_dense shape  : {X_dense.shape}")
print(f"medium_tfidf   : {medium_tfidf.shape}")
print(f"tags_tfidf     : {tags_tfidf.shape}")
print(f"period_tfidf   : {period_tfidf.shape}")
print(f"X final shape  : {X.shape}")

X_dense shape  : (484956, 14)
medium_tfidf   : (484956, 100)
tags_tfidf     : (484956, 50)
period_tfidf   : (484956, 50)
X final shape  : (484956, 214)


In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df["Department"])

print(f"y shape: {y.shape}")
print()
print("Class mapping (Department → integer label):")
print(dict(zip(le.classes_, le.transform(le.classes_))))

y shape: (484956,)

Class mapping (Department → integer label):
{'Ancient Near Eastern Art': np.int64(0), 'Arms and Armor': np.int64(1), 'Arts of Africa, Oceania, and the Americas': np.int64(2), 'Asian Art': np.int64(3), 'Costume Institute': np.int64(4), 'Drawings and Prints': np.int64(5), 'Egyptian Art': np.int64(6), 'European Paintings': np.int64(7), 'European Sculpture and Decorative Arts': np.int64(8), 'Greek and Roman Art': np.int64(9), 'Islamic Art': np.int64(10), 'Medieval Art': np.int64(11), 'Modern and Contemporary Art': np.int64(12), 'Musical Instruments': np.int64(13), 'Photographs': np.int64(14), 'Robert Lehman Collection': np.int64(15), 'The American Wing': np.int64(16), 'The Cloisters': np.int64(17), 'The Libraries': np.int64(18)}


### Train / Test Split

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

# ── Stratification check ──────────────────────────────────────────────────────
# Class proportions in train and test should be nearly identical.
# A large gap in any row means stratification failed for that class.

n_classes = len(le.classes_)
train_props = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_props  = pd.Series(y_test).value_counts(normalize=True).sort_index()

check = pd.DataFrame({
    "Department"  : le.classes_,
    "train_pct"   : (train_props.values * 100).round(2),
    "test_pct"    : (test_props.values * 100).round(2),
})
check["diff"] = (check["train_pct"] - check["test_pct"]).abs().round(3)

display(check.sort_values("train_pct", ascending=False).head(5))
print()
print(f"Max proportion difference across all classes: {check['diff'].max():.4f}%")

X_train : (387964, 214)
X_test  : (96992, 214)
y_train : (387964,)
y_test  : (96992,)


,Department,train_pct,test_pct,diff
5,Drawings and Prints,35.60,35.60,0.0
8,European Sculpture and Decorative Arts,8.88,8.88,0.0
14,Photographs,7.72,7.72,0.0
3,Asian Art,7.63,7.63,0.0
9,Greek and Roman Art,6.95,6.95,0.0



Max proportion difference across all classes: 0.0000%


### Save Artifacts

In [24]:
import joblib
from scipy.sparse import save_npz

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
MODELS_DIR    = os.path.join(ROOT, "models")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Sparse matrices — .npz preserves sparsity
save_npz(os.path.join(PROCESSED_DIR, "X_train.npz"), X_train)
save_npz(os.path.join(PROCESSED_DIR, "X_test.npz"),  X_test)

# Dense label arrays — .npy is numpy's native binary format
np.save(os.path.join(PROCESSED_DIR, "y_train.npy"), y_train)
np.save(os.path.join(PROCESSED_DIR, "y_test.npy"),  y_test)

# LabelEncoder — needed to convert integer predictions back to department names
joblib.dump(le, os.path.join(MODELS_DIR, "label_encoder.joblib"))

print("Saved:")
print(f"  {PROCESSED_DIR}/X_train.npz")
print(f"  {PROCESSED_DIR}/X_test.npz")
print(f"  {PROCESSED_DIR}/y_train.npy")
print(f"  {PROCESSED_DIR}/y_test.npy")
print(f"  {MODELS_DIR}/label_encoder.joblib")

Saved:
  /Users/ishan/Documents/Northeastern/Spring 2026/Machine Learning/Final project/cultural-pattern-discovery-ml/data/processed/X_train.npz
  /Users/ishan/Documents/Northeastern/Spring 2026/Machine Learning/Final project/cultural-pattern-discovery-ml/data/processed/X_test.npz
  /Users/ishan/Documents/Northeastern/Spring 2026/Machine Learning/Final project/cultural-pattern-discovery-ml/data/processed/y_train.npy
  /Users/ishan/Documents/Northeastern/Spring 2026/Machine Learning/Final project/cultural-pattern-discovery-ml/data/processed/y_test.npy
  /Users/ishan/Documents/Northeastern/Spring 2026/Machine Learning/Final project/cultural-pattern-discovery-ml/models/label_encoder.joblib


In [25]:
from scipy.sparse import load_npz

X_train_check = load_npz(os.path.join(PROCESSED_DIR, "X_train.npz"))
X_test_check  = load_npz(os.path.join(PROCESSED_DIR, "X_test.npz"))
y_train_check = np.load(os.path.join(PROCESSED_DIR, "y_train.npy"))
y_test_check  = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))
le_check      = joblib.load(os.path.join(MODELS_DIR, "label_encoder.joblib"))

assert X_train_check.shape == X_train.shape, "X_train shape mismatch"
assert X_test_check.shape  == X_test.shape,  "X_test shape mismatch"
assert y_train_check.shape == y_train.shape, "y_train shape mismatch"
assert y_test_check.shape  == y_test.shape,  "y_test shape mismatch"
assert list(le_check.classes_) == list(le.classes_), "LabelEncoder classes mismatch"

print(f"X_train : {X_train_check.shape}")
print(f"X_test  : {X_test_check.shape}")
print(f"y_train : {y_train_check.shape}")
print(f"y_test  : {y_test_check.shape}")
print(f"LabelEncoder classes: {le_check.classes_.tolist()}")
print()
print("Round-trip verification passed ✓")

X_train : (387964, 214)
X_test  : (96992, 214)
y_train : (387964,)
y_test  : (96992,)
LabelEncoder classes: ['Ancient Near Eastern Art', 'Arms and Armor', 'Arts of Africa, Oceania, and the Americas', 'Asian Art', 'Costume Institute', 'Drawings and Prints', 'Egyptian Art', 'European Paintings', 'European Sculpture and Decorative Arts', 'Greek and Roman Art', 'Islamic Art', 'Medieval Art', 'Modern and Contemporary Art', 'Musical Instruments', 'Photographs', 'Robert Lehman Collection', 'The American Wing', 'The Cloisters', 'The Libraries']

Round-trip verification passed ✓


### Pipeline — Build, Fit, Save

In [26]:
import sys
sys.path.insert(0, ROOT)          # make the repo root importable
from src import build_pipeline

# ── Split df_raw into train / test BEFORE encoding ────────────────────────────
# The pipeline must be fit only on training data. We therefore split the raw
# dataframe (not the already-encoded df) using the same seed and stratification
# so class proportions match the earlier split.

y_dept   = LabelEncoder().fit_transform(df_raw["Department"])
X_raw_df = df_raw.drop(columns=["Department"])

train_idx, test_idx = train_test_split(
    np.arange(len(X_raw_df)), test_size=0.2, random_state=42, stratify=y_dept
)

df_train_raw = X_raw_df.iloc[train_idx].reset_index(drop=True)
df_test_raw  = X_raw_df.iloc[test_idx].reset_index(drop=True)

print(f"df_train_raw : {df_train_raw.shape}")
print(f"df_test_raw  : {df_test_raw.shape}")

df_train_raw : (387964, 53)
df_test_raw  : (96992, 53)


In [27]:
pipeline = build_pipeline()

X_train_pipe = pipeline.fit_transform(df_train_raw)   # fit on train only
X_test_pipe  = pipeline.transform(df_test_raw)         # transform with frozen state

print(f"X_train_pipe : {X_train_pipe.shape}")
print(f"X_test_pipe  : {X_test_pipe.shape}")

joblib.dump(pipeline, os.path.join(MODELS_DIR, "preprocessing_pipeline.joblib"))
print(f"Pipeline saved → models/preprocessing_pipeline.joblib")

X_train_pipe : (387964, 213)
X_test_pipe  : (96992, 213)
Pipeline saved → models/preprocessing_pipeline.joblib
